In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [2]:
# Load the dataset from the CSV file into a pandas DataFrame
data = pd.read_csv("../Data/Churn_Modelling.csv")

# Display the first five rows of the dataset to preview its structure
data.head()


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
## Preprocess the data
### Drop irrelevant columns
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)#axis=1 this mean column vise
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [4]:
## Encode categorical variables

# Create a LabelEncoder instance for the 'Gender' column
label_encoder_gender = LabelEncoder()

# Fit the encoder on the 'Gender' column and transform it into numeric values
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [5]:
# we can not apply label encoder  on the Geography column because there are three values so for that we are using OneHotEncoder

## Onehot encode 'Geography
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo=OneHotEncoder()
geo_encoder=onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoder

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [6]:
# Get the names of the one-hot encoded features for the 'Geography' column
onehot_encoder_geo.get_feature_names_out(['Geography'])


array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [7]:
# Create a new DataFrame from the one-hot encoded 'Geography' values
# 'geo_encoder' contains the transformed array
# 'onehot_encoder_geo.get_feature_names_out()' gives proper column names
geo_encoded_df = pd.DataFrame(
    geo_encoder,
    columns = onehot_encoder_geo.get_feature_names_out(['Geography'])
)

# Display the one-hot encoded DataFrame
geo_encoded_df


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [8]:
## Combine one-hot encoded columns with the original data

# Remove the original 'Geography' column and add the new one-hot encoded columns
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

# Show the first few rows of the updated dataset
data.head()


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [9]:
## Save the encoders

# Save the label encoder for 'Gender' column to a file
# This allows you to reuse it later for transforming new data
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

# Save the one-hot encoder for 'Geography' column to a file
# This allows consistent transformation for new data in the future
with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)


In [10]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [11]:
## Divide the dataset into independent and dependent features

# X contains all features except the target column 'Exited'
X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)

# Transform the test data using the same scaler (do NOT fit again) to check why we don't fit to test data i have written under 
X_test = scaler.transform(X_test)

scaler


,copy,True
,with_mean,True
,with_std,True


In [12]:
X_train

array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
         1.72572313, -0.57638802]])

In [13]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


### ANN Implementation

In [14]:
import sys
print(sys.executable)


c:\Users\ADMIN\miniconda3\envs\ann_env\python.exe


In [15]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [16]:
(X_train.shape[1],)#number of features(columns) after removal of target column

(12,)

In [17]:
## Build Our ANN Model

model = Sequential([
    # Hidden Layer 1: 64 neurons, ReLU activation, connected to input features
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),  
    # input_shape=(X_train.shape[1],) #tells Keras how many inputs each neuron in the first layer should expect.
    
    # Hidden Layer 2: 32 neurons, ReLU activation
    Dense(32, activation='relu'),  
    
    # Output Layer: 1 neuron with Sigmoid activation for binary classification
    Dense(1, activation='sigmoid')  
])


In [18]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [19]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)#ANN training will fail. without this 
loss=tensorflow.keras.losses.BinaryCrossentropy()
loss


In [20]:
## Compile the model (without this learning can not be start) 
model.compile(
    optimizer=opt,
    loss="binary_crossentropy",     metrics=['accuracy']
)


In [21]:

log_dir="logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
# Creates a folder to save logs for TensorBoard.
# datetime.datetime.now().strftime("%Y%m%d-%H%M%S") adds a timestamp, so each run gets a unique folder.
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [22]:
## Set up the Tensorboard
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

## Set up Early Stopping
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)
# EarlyStopping is a Keras callback that stops training before all epochs are completed if the model stops improving.


In [23]:
### Train the model
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

Epoch 1/100


250/250 [==============================] - 2s 4ms/step - loss: 0.3975 - accuracy: 0.8304 - val_loss: 0.3533 - val_accuracy: 0.8540
Epoch 2/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3553 - accuracy: 0.8546 - val_loss: 0.3418 - val_accuracy: 0.8565
Epoch 3/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3465 - accuracy: 0.8579 - val_loss: 0.3453 - val_accuracy: 0.8575
Epoch 4/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3415 - accuracy: 0.8631 - val_loss: 0.3453 - val_accuracy: 0.8575
Epoch 5/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3404 - accuracy: 0.8629 - val_loss: 0.3465 - val_accuracy: 0.8530
Epoch 6/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3387 - accuracy: 0.8624 - val_loss: 0.3445 - val_accuracy: 0.8575
Epoch 7/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3345 - accuracy: 0.8599 - val_loss: 0.3585 - val_accuracy: 0.85

In [24]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs > 0.5).astype(int)

print("Predicted probabilities (first 10):")
print(y_pred_probs[:10])

print("\nPredicted classes (first 10):")
print(y_pred[:10])

print("\nActual labels (first 10):")
print(y_test.values[:10])

print("\nClassification report:")
print(classification_report(y_test, y_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

63/63 [==============================] - 0s 760us/step
Predicted probabilities (first 10):
[[0.02797853]
 [0.02263253]
 [0.03462928]
 [0.30548283]
 [0.06089935]
 [0.00348645]
 [0.13524118]
 [0.21864721]
 [0.11653817]
 [0.4119786 ]]

Predicted classes (first 10):
[[0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]]

Actual labels (first 10):
[0 0 0 0 0 0 0 1 0 0]

Classification report:
              precision    recall  f1-score   support

           0       0.88      0.95      0.91      1607
           1       0.69      0.48      0.56       393

    accuracy                           0.85      2000
   macro avg       0.79      0.71      0.74      2000
weighted avg       0.84      0.85      0.84      2000


Confusion matrix:
[[1523   84]
 [ 206  187]]


In [25]:
sample = X_test[:22]
sample_pred = model.predict(sample)
predicted_classes = (sample_pred > 0.5).astype(int)

print("Predicted classes for first 22 rows:")
print(predicted_classes.flatten())

1/1 [==============================] - 0s 19ms/step
Predicted classes for first 22 rows:
[0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0]


In [26]:
for i, prob in enumerate(sample_pred, start=1):
    print(i, "prob:", prob[0], "class:", int(prob[0] > 0.5))

1 prob: 0.027978528 class: 0
2 prob: 0.022632528 class: 0
3 prob: 0.034629278 class: 0
4 prob: 0.30548283 class: 0
5 prob: 0.060899355 class: 0
6 prob: 0.0034864529 class: 0
7 prob: 0.13524118 class: 0
8 prob: 0.21864721 class: 0
9 prob: 0.116538174 class: 0
10 prob: 0.4119786 class: 0
11 prob: 0.95659584 class: 1
12 prob: 0.9163718 class: 1
13 prob: 0.49783003 class: 0
14 prob: 0.39754388 class: 0
15 prob: 0.008366232 class: 0
16 prob: 0.17367437 class: 0
17 prob: 0.115782425 class: 0
18 prob: 0.059687365 class: 0
19 prob: 0.006388276 class: 0
20 prob: 0.04048164 class: 0
21 prob: 0.10778295 class: 0
22 prob: 0.014030026 class: 0


In [27]:
model.save('model.h5')

c:\Users\ADMIN\miniconda3\envs\ann_env\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [28]:
def risk_category(prob):
    if prob > 0.7:
        return "High Risk 🔴"
    elif prob > 0.4:
        return "Medium Risk 🟡"
    else:
        return "Low Risk 🟢"

In [29]:
risk_labels = [risk_category(p[0]) for p in y_pred_probs]

In [30]:
import pandas as pd

results_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted_Prob": y_pred_probs.flatten(),
    "Risk_Level": risk_labels
})

print(results_df.head(10))

   Actual  Predicted_Prob     Risk_Level
0       0        0.027979     Low Risk 🟢
1       0        0.022633     Low Risk 🟢
2       0        0.034629     Low Risk 🟢
3       0        0.305483     Low Risk 🟢
4       0        0.060899     Low Risk 🟢
5       0        0.003486     Low Risk 🟢
6       0        0.135241     Low Risk 🟢
7       1        0.218647     Low Risk 🟢
8       0        0.116538     Low Risk 🟢
9       0        0.411979  Medium Risk 🟡


In [31]:
print("High Risk Customers:", sum(results_df['Risk_Level'] == "High Risk 🔴"))
print("Medium Risk Customers:", sum(results_df['Risk_Level'] == "Medium Risk 🟡"))
print("Low Risk Customers:", sum(results_df['Risk_Level'] == "Low Risk 🟢"))

High Risk Customers: 158
Medium Risk Customers: 179
Low Risk Customers: 1663


In [32]:
## Load Tensorboard Extension
%load_ext tensorboard